# Overlay de eventos (espectador) — demo reproducible desde `src`

Reproduce el **video demo de espectador** llamando **solo a `src`** (no a código de
notebook), de modo que el mismo pipeline aplica a **cualquier** clip de cámara superior.
El clip de referencia es `IMG_9933_5m30` (el del notebook
`fase_4_homografia/v2_07_minimap_polish_cenital.ipynb`).

Sobre la ronda anterior, el overlay ahora **adopta la homografía actual (por líneas,
`VideoHomographyLines`)** consolidada en `src`:

- **homografía embebida**: el campo (rectángulo interior + círculo central) se reproyecta
  sobre el video real con la H por frame;
- **minimapa cenital** (`CenitalMinimapRenderer`) y **heatmap** con el mismo estilo
  (cancha vertical, porterías por color), en el margen derecho;
- marcador 0-0 por color de portería (sube con cada gol), banner *¡Goool!* deslizante,
  métricas de posesión/control en el margen izquierdo y lista dinámica de eventos con tope.

### Dónde corre (pod vs. local)

- La **detección pesada** (SAM3 + YOLO → `tracks_json` con `include_masks=True`) es trabajo
  **de pod** (GPU), ya hecho: este notebook **no** la re-ejecuta.
- El **render del overlay** corre en **CPU local** consumiendo el `tracks_json` + el `.mp4`
  (la homografía por líneas se reconstruye desde la carpet-mask del JSON + los píxeles del
  clip). Visualiza el resultado aquí mismo.

## 0 — Setup y rutas (config-driven)

Las rutas se resuelven con `src.utils.get_abs_path` contra `PROJECT_ROOT` (no se
hardcodea nada). Sustituye `CLIP_JSON_REL` para correr el overlay sobre otro clip.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from src.utils import PROJECT_ROOT, get_abs_path
from src.core.event_broadcast_overlay import render_broadcast_overlay

# tracks_json del clip de referencia (cámara superior). El .mp4 se busca junto al JSON.
CLIP_JSON_REL = "outputs/inference/fase5_clips/IMG_9933_5m30/IMG_9933_5m30.json"
clip_json = get_abs_path(CLIP_JSON_REL)
clip_mp4 = clip_json.with_suffix(".mp4")
print("JSON:", clip_json.relative_to(PROJECT_ROOT))
print("MP4 :", clip_mp4.relative_to(PROJECT_ROOT), "| existe:", clip_mp4.exists())

## 1 — Render del overlay

Parámetros principales: `layout` (1 paneles superpuestos | 2 paneles laterales con
márgenes, default), `goal_source` (`strict` | `geometric`), `draw_field_on_video`
(homografía embebida, ON).

**`MAX_FRAMES`**: déjalo en `None` para el **demo completo** (~1799 frames, varios minutos);
pon p. ej. `120` para una **vista rápida** de prueba.

In [ ]:
MAX_FRAMES = None  # None = demo completo; o un entero (p. ej. 120) para vista rápida

res = render_broadcast_overlay(
    clip_json,
    layout=2,
    goal_source="strict",
    draw_field_on_video=True,
    max_frames=MAX_FRAMES,
    progress=True,
)
res.resumen

## 2 — Inspección visual

`overlay_degradado` debe ser `False` (la homografía por líneas funcionó). Mostramos un
frame compuesto: si hubo gol, el PNG de muestra (con banner); si no, un frame del centro
del mp4 de salida.

In [ ]:
if res.sample_png is not None and Path(res.sample_png).exists():
    frame_bgr = cv2.imread(str(res.sample_png))
    titulo = f"sample (gol): {Path(res.sample_png).name}"
else:
    cap = cv2.VideoCapture(str(res.video))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2)
    _ok, frame_bgr = cap.read()
    cap.release()
    titulo = f"frame central de {Path(res.video).name}"

plt.figure(figsize=(11, 9))
plt.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
plt.title(titulo)
plt.axis("off")
plt.show()
print("video:", Path(res.video).relative_to(PROJECT_ROOT))

In [ ]:
# Reproducción inline del mp4 (opcional; puede no renderizar en todos los entornos).
from IPython.display import Video

Video(str(res.video), embed=False, width=640)

## 3 — Requisitos cubiertos

- [x] **Márgenes** alrededor del video (layout 2).
- [x] **Marcador 0-0** por color de portería; sube con cada gol.
- [x] **Banner** *¡Goool! Portería {color}* deslizante.
- [x] **Métricas** de posesión/control en el **margen izquierdo**.
- [x] **Lista dinámica** de eventos (tiros, fueras, pushing, sin progreso) con **tope**.
- [x] **Minimapa cenital** + **heatmap** (mismo estilo) en el margen derecho.
- [x] **Homografía actual (por líneas)** embebida sobre el video.

Para otro clip: cambia `CLIP_JSON_REL` por el `tracks_json` (con `include_masks=True`) de
otro video de **cámara superior** y reejecuta.